# Initial data inspection and basket exploration

Purpose: understand the raw data and document the initial basket choices. No price cleaning is applied. Saved outputs have been cleared to keep the file small; run the cells to regenerate tables. The CSV reports remain in your project.

## 1. Setup

Run from top to bottom. Uses the already downloaded August and September 2026 files; these are exploratory months, not the final analysis period.

In [ ]:
from pathlib import Path
from itertools import combinations
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

# Find the project when Jupyter starts in its root or notebooks folder.
cwd = Path.cwd().resolve()
project_root = next(
    (folder for folder in [cwd, *cwd.parents]
     if (folder / "src" / "check_env.py").is_file()),
    Path(r"C:\Users\User\OneDrive\Desktop\malaysia-food-prices"),
)
assert (project_root / "src" / "check_env.py").is_file(), "Set project_root to your project folder."
raw_dir = project_root / "data" / "raw"
docs_dir = project_root / "docs"
docs_dir.mkdir(parents=True, exist_ok=True)

months_needed = ['2026-08', '2026-09']
required = ["lookup_item.parquet", "lookup_premise.parquet"] + [
    f"pricecatcher_{month}.parquet" for month in months_needed
]
missing = [name for name in required if not (raw_dir / name).is_file()]
assert not missing, f"Missing local inputs: {missing}"
print("Project:", project_root)
print("Required local files found:", len(required))

## 2. Inspect schemas and columns

Check file structure, sample rows, missing values and lookup keys.

In [ ]:
files = ["pricecatcher_2026-09.parquet", "lookup_item.parquet", "lookup_premise.parquet"]
for filename in files:
    parquet_file = pq.ParquetFile(raw_dir / filename)

    print(f"\n--- {filename} ---")
    print("Schema:")
    print(parquet_file.schema_arrow)

    # Read a small batch instead of loading the entire file.
    sample = next(parquet_file.iter_batches(batch_size=5))
    display(sample.to_pandas())

prices = pd.read_parquet(raw_dir/ "pricecatcher_2026-09.parquet")
items = pd.read_parquet(raw_dir/ "lookup_item.parquet")
premises = pd.read_parquet(raw_dir/ "lookup_premise.parquet")

tables = {
    "prices":prices,
    "items":items,
    "premises":premises,
}

for name, df in tables.items():
    print(f"\n--- {name}: {len(df):,} rows ---")

    profile = pd.DataFrame({
         "dtype": df.dtypes.astype(str),
         "null_count": df.isna().sum(),
         "distinct_non_null": df.nunique(dropna=True),
    })

    print(profile.to_string())
    
for name, df, key in [
    ("items", items, "item_code"),
    ("premises", premises, "premise_code"),
]:
    print(f"\n{name}:")
    print("Missing lookup keys:", df[key].isna().sum())
    print("Repeated keys beyond first:", df[key].duplicated().sum())

print("\nPrice records using item_code -1:",
      prices["item_code"].eq(-1).sum())
print("Price records using premise_code -1:",
      prices["premise_code"].eq(-1).sum())

## 3. Check September dates and lookup matches

Dates without observations are gaps, not zero prices. Lookup matches tell us whether names and locations can be attached.

In [ ]:
dates = pd.to_datetime(prices["date"], errors="raise")

expected_dates = pd.date_range("2026-09-01", "2026-09-30")
observed_dates = pd.DatetimeIndex(dates.dt.normalize().unique())

print("Earliest date:", dates.min())
print("Latest date:", dates.max())

missing_dates = expected_dates.difference(observed_dates)
outside_dates = observed_dates.difference(expected_dates)

print("\nSeptember dates without records:")
print(missing_dates.strftime("%Y-%m-%d").tolist())

print("\nDates outside September:")
print(outside_dates.strftime("%Y-%m-%d").tolist())

daily_counts = (
    dates.dt.normalize()
    .value_counts()
    .sort_index()
    .reindex(expected_dates, fill_value=0)
)

print("\nRecords per September date:")
print(daily_counts.to_string())
print("\nTotal September records:", daily_counts.sum())

for key, lookup in [
    ("item_code", items),
    ("premise_code", premises),
]:
    matched = prices[key].isin(lookup[key])

    print(f"\n--- {key} lookup coverage ---")
    print("Matched records:", f"{matched.sum():,}")
    print("Unmatched records:", f"{(~matched).sum():,}")
    print("Matched percentage:", f"{matched.mean():.2%}")

    unmatched_codes = (
        prices.loc[~matched, key]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )
    print("Unmatched codes:", unmatched_codes)

## 4. Review prices and duplicates

Extreme values are candidates for investigation, not automatic corrections. Prices across different products are not directly comparable.

In [ ]:
print("Price summary across all items:")
print(
    prices["price"]
    .quantile([0, 0.01, 0.50, 0.99, 1])
    .rename(index={
        0: "min",
        0.01: "p1",
        0.50: "median",
        0.99: "p99",
        1: "max",
    })
    .to_string()
)

print("\nZero prices:", prices["price"].eq(0).sum())
print("Negative prices:", prices["price"].lt(0).sum())

print(
    "\nExact duplicate rows beyond first:",
    prices.duplicated().sum()
)

candidate_key = ["date", "premise_code", "item_code"]

print(
    "Repeated date-premise-item combinations beyond first:",
    prices.duplicated(subset=candidate_key).sum()
)

item_details = items[["item_code", "item", "unit"]]

for label, sample in [
    ("10 lowest prices", prices.nsmallest(10, "price")),
    ("10 highest prices", prices.nlargest(10, "price")),
]:
    detailed = sample.merge(
        item_details,
        on="item_code",
        how="left",
        validate="many_to_one",
    )

    print(f"\n--- {label} ---")
    print(detailed.to_string(index=False))

# Collect the item codes appearing in the extreme-price samples.
flagged_codes = pd.concat([
    prices.nsmallest(10, "price")["item_code"],
    prices.nlargest(10, "price")["item_code"],
]).unique()

# Summarise this month's observations for each flagged item.
item_summary = (
    prices.loc[prices["item_code"].isin(flagged_codes)]
    .groupby("item_code")["price"]
    .agg(
        observations="count",
        minimum="min",
        median="median",
        maximum="max",
    )
    .reset_index()
    .merge(
        items[["item_code", "item", "unit"]],
        on="item_code",
        how="left",
        validate="many_to_one",
    )
)

print(item_summary.to_string(index=False))

docs_dir.mkdir(parents=True, exist_ok=True)

output_path = docs_dir / "extreme_item_summary.csv"
item_summary.to_csv(output_path, index=False)

saved = pd.read_csv(output_path)

print("Saved:", output_path)
print("Rows:", len(saved))
print("Item codes unique:", saved["item_code"].is_unique)

## 5. Review the original basket

This is the original selection, retained to explain later replacements. Notebook 02 uses the revised basket.

In [ ]:
basket_codes = [
    1, 1111, 992, 918, 1589,
    1593, 129, 1564, 1131, 94,
    114, 103, 1370, 1555, 193,
]

prices = pd.read_parquet(raw_dir / "pricecatcher_2026-09.parquet")
items = pd.read_parquet(raw_dir / "lookup_item.parquet")
premises = pd.read_parquet(raw_dir / "lookup_premise.parquet")

# Keep only the selected items, then attach location information.
selected = prices.loc[prices["item_code"].isin(basket_codes)]

selected = selected.merge(
    premises[["premise_code", "state", "premise_type"]],
    on="premise_code",
    how="left",
    validate="many_to_one",
)

coverage = (
    selected.groupby("item_code")
    .agg(
        observations=("price", "size"),
        recorded_days=("date", "nunique"),
        premises=("premise_code", "nunique"),
        states=("state", "nunique"),
        premise_types=("premise_type", "nunique"),
    )
    .reset_index()
)

# Start from all 15 selected codes so absent items remain visible.
basket = pd.DataFrame({"item_code": basket_codes})

coverage = (
    basket.merge(
        items[["item_code", "item", "unit"]],
        on="item_code",
        how="left",
        validate="one_to_one",
    )
    .merge(
        coverage,
        on="item_code",
        how="left",
        validate="one_to_one",
    )
)

count_columns = [
    "observations", "recorded_days", "premises",
    "states", "premise_types",
]
coverage[count_columns] = coverage[count_columns].fillna(0).astype(int)

print(coverage.to_string(index=False))
print("\nBasket rows:", len(coverage))
print("Total selected records:", coverage["observations"].sum())
print("Source dates available:", prices["date"].nunique())

docs_dir.mkdir(parents=True, exist_ok=True)

output_path = docs_dir / "basket_coverage_2026-09.csv"
coverage.to_csv(output_path, index=False, encoding="utf-8-sig")

saved = pd.read_csv(output_path)

print("Saved:", output_path)
print("Basket rows:", len(saved))
print("Item codes unique:", saved["item_code"].is_unique)
print("Total selected records:", saved["observations"].sum())

## 6. Compare potential replacements

Observation counts describe coverage, not consumption. Save the candidate evidence.

In [ ]:
# Find potentially relevant items in the lookup.
mask = items["item"].str.contains(
    r"BERAS|TEPUNG GANDUM|DAGING LEMBU",
    case=False,
    na=False,
)

candidates = items.loc[mask, ["item_code", "item", "unit"]].copy()

# Measure coverage for each item in the current price file.
counts = (
    prices.groupby("item_code")
    .agg(
        observations=("price", "size"),
        recorded_days=("date", "nunique"),
        premises=("premise_code", "nunique"),
    )
    .reset_index()
)

candidate_coverage = candidates.merge(
    counts,
    on="item_code",
    how="left",
    validate="one_to_one",
)

count_columns = ["observations", "recorded_days", "premises"]
candidate_coverage[count_columns] = (
    candidate_coverage[count_columns].fillna(0).astype(int)
)

candidate_coverage = candidate_coverage.sort_values(
    ["recorded_days", "observations"],
    ascending=False,
)

print(candidate_coverage.to_string(index=False))
print("\nCandidate items:", len(candidate_coverage))
print("Item codes unique:", candidate_coverage["item_code"].is_unique)

output_path = project_root / "docs" / "basket_candidates_2026-09.csv"
candidate_coverage.to_csv(output_path, index=False, encoding="utf-8-sig")

saved = pd.read_csv(output_path)
print("Saved:", output_path)
print("Candidate rows:", len(saved))
print("Item codes unique:", saved["item_code"].is_unique)

## 7. Check August and compare the two exploratory months

Keep the original and alternative codes visible. Historical checks in Notebook 02 determine whether coverage persists.

In [ ]:
destination = raw_dir / "pricecatcher_2026-08.parquet"
august_dates = pd.read_parquet(destination, columns=["date"])
dates = pd.to_datetime(august_dates["date"], errors="raise")

expected = pd.date_range("2026-08-01", "2026-08-31")
observed = pd.DatetimeIndex(dates.dropna().dt.normalize().unique())

print("\nRows:", f"{len(dates):,}")
print("Earliest date:", dates.min())
print("Latest date:", dates.max())
print("Distinct dates:", len(observed))
print("Missing date values:", dates.isna().sum())
print("August dates without records:",
      expected.difference(observed).strftime("%Y-%m-%d").tolist())
print("Dates outside August:",
      observed.difference(expected).strftime("%Y-%m-%d").tolist())

# Original basket plus alternative rice, flour, and beef.
review_codes = [
    1, 1111, 992, 918, 1589,
    1593, 129, 1564, 1131, 94,
    114, 103, 1370, 1555, 193,
    2066, 917, 1431,
]

review_items = pd.DataFrame({"item_code": review_codes}).merge(
    items[["item_code", "item", "unit"]],
    on="item_code",
    how="left",
    validate="one_to_one",
)

results = []

for month in ["2026-08", "2026-09"]:
    monthly = pd.read_parquet(
        raw_dir / f"pricecatcher_{month}.parquet"
    )
    source_days = monthly["date"].nunique()

    selected = monthly.loc[monthly["item_code"].isin(review_codes)]

    summary = (
        selected.groupby("item_code")
        .agg(
            observations=("price", "size"),
            recorded_days=("date", "nunique"),
            premises=("premise_code", "nunique"),
        )
        .reset_index()
    )

    summary = review_items.merge(
        summary,
        on="item_code",
        how="left",
        validate="one_to_one",
    )

    count_columns = ["observations", "recorded_days", "premises"]
    summary[count_columns] = (
        summary[count_columns].fillna(0).astype(int)
    )

    summary["month"] = month
    summary["source_days"] = source_days
    results.append(summary)

comparison = pd.concat(results, ignore_index=True)

print(
    comparison[
        ["item_code", "month", "observations",
         "recorded_days", "source_days", "premises"]
    ]
    .sort_values(["item_code", "month"])
    .to_string(index=False)
)

print("\nComparison rows:", len(comparison))
print(
    "Item-month pairs unique:",
    not comparison.duplicated(["item_code", "month"]).any(),
)

output_path = project_root / "docs" / "basket_coverage_aug_sep_2026.csv"
comparison.to_csv(output_path, index=False, encoding="utf-8-sig")

saved = pd.read_csv(output_path)

print("Saved:", output_path)
print("Rows:", len(saved))
print(
    "Item-month pairs unique:",
    not saved.duplicated(["item_code", "month"]).any(),
)